In [57]:
#### IMPORTS
import numpy as np

In [104]:
#### UTILITIES 
def create_adjacancy__matrix(block_sizes):
    """
    Create a block matrix with ones on the diagonal blocks and zeros elsewhere.
    
    Parameters:
    block_sizes (list of int): Sizes of the blocks along the diagonal.
    
    Returns:
    np.ndarray: The resulting block matrix.
    """

    K = len(block_sizes)
    # Calculate the total size of the matrix
    total_size = sum(block_sizes)
    # Initialize the full matrix with zeros
    block_matrix = np.zeros((total_size, total_size), dtype=int)
    
    # Fill the diagonal blocks with ones
    start_index = 0
    for size in block_sizes:
        block_matrix[start_index:start_index + size, start_index:start_index + size] = 1
        start_index += size
    
    eigenvalues, eigenvectors = np.linalg.eigh(block_matrix)
    top_indices = np.argsort(eigenvalues)[-K:]  # Indices of the K largest eigenvalues
    Lambda = eigenvalues[top_indices]   # The K largest eigenvalues
    E = eigenvectors[:, top_indices] # The eigenvectors corresponding to the largest eigenvalues, the dimension is nxK 

    block_bar = block_matrix.copy()
    block_bar[block_bar == 0] = -1

    return block_matrix, block_bar, Lambda, E

def create_Ri_symmetric_matrix_with_constrained_eigenvalues(dimension, rho_minus, rho_plus, noise_level=0.0, include_extremes=True):
    """
    Create a symmetric matrix with eigenvalues between rho_minus and rho_plus.
    
    Parameters:
    dimension (int): Size of the matrix (n x n).
    rho_minus (float): Lower bound of the eigenvalues.
    rho_plus  (float): Upper bound of the eigenvalues.
    noise_level (float): The scale of the noise to be added (small value to slightly perturb).
    include_extremes (bool): If True, include rho_minus and rho_plus as exact eigenvalues.

    Returns:
    np.ndarray: The symmetric matrix with eigenvalues in the range [rho_minus, rho_plus].
    """
    if include_extremes:
        if dimension < 2:
            raise ValueError("Dimension must be at least 2 to include both rho_minus and rho_plus as eigenvalues.")
        # Generate the remaining dimension - 2 eigenvalues uniformly between rho_minus and rho_plus
        additional_eigenvalues = np.random.uniform(rho_minus, rho_plus, dimension - 2)
        # Combine all eigenvalues
        all_eigenvalues = np.concatenate(([rho_minus, rho_plus], additional_eigenvalues))
    else:
        # Generate dimension eigenvalues uniformly distributed between rho_minus and rho_plus
        all_eigenvalues = np.random.uniform(rho_minus, rho_plus, dimension)
    
    # Create rho_minus random orthogonal matrix Q using QR decomposition
    Q, _ = np.linalg.qr(np.random.randn(dimension, dimension))

    # Create symmetric noise matrix
    noise_matrix = noise_level*np.random.randn(dimension,dimension)
    symmetric_noise = (noise_matrix + noise_matrix.T) / 2

    # Construct the symmetric matrix using A = Q * Lambda * Q.T
    Lambda = np.diag(all_eigenvalues)  # Diagonal matrix of eigenvalues
    symmetric_matrix = Q @ Lambda @ Q.T + symmetric_noise
    
    return symmetric_matrix

def create_multiple_noisy_probability_vectors(count, dimension, noise_level=0.0, manual_probabilities=None):
    """
    Create multiple noisy probability vectors where each starts with a unique index set to 1.
    
    Parameters:
    count (int): Number of probability vectors to generate.(In our problem: the number of clusters)
    dimension (int): Length of each probability vector.(In our probelm: the number of classes)
    noise_level (float): The scale of the noise to add.
    manual_probabilities (list of lists or None): If provided, use this as the probability vectors.

    Returns:
    np.ndarray: An array containing the noisy probability vectors.
    """

    # If manual probabilities are provided, return them as a numpy array
    if manual_probabilities is not None:
        # Check that manual probabilities have the correct shape
        manual_probabilities = np.array(manual_probabilities)
        if manual_probabilities.shape != (count, dimension):
            raise ValueError(f"manual_probabilities must have shape ({count}, {dimension})")
        
        # Ensure that each row sums to 1 to confirm they are valid probability vectors
        if not np.allclose(np.sum(manual_probabilities, axis=1), 1):
            raise ValueError("Each probability vector must sum to 1")
        
        return manual_probabilities
    
    # Check that count does not exceed dimension
    if count > dimension:
        raise ValueError("The number of vectors 'count' cannot exceed the dimension size.")
    
    probability_vectors = []
    
    # Step 1: Create 'count' one-hot vectors at unique indices
    for i in range(count):
        # Initialize a one-hot vector with 1 at index i
        probability_vector = np.zeros(dimension)
        probability_vector[i] = 1
        
        # Step 2: Add random noise to the vector
        noise = np.random.uniform(0, noise_level, dimension)
        noisy_vector = probability_vector + noise
        
        # Step 3: Normalize to make it a probability vector
        noisy_probability_vector = noisy_vector / np.sum(noisy_vector)
        
        probability_vectors.append(noisy_probability_vector)
    
    return np.array(probability_vectors)

def project_to_set_S(matrix, eigen_min, eigen_max):
    """
    Project a matrix onto the set of symmetric matrices with eigenvalues
    constrained between eigen_min and eigen_max.
    
    Parameters:
    matrix (np.ndarray): The input matrix to be projected.
    eigen_min (float): The minimum eigenvalue bound.
    eigen_max (float): The maximum eigenvalue bound.
    
    Returns:
    np.ndarray: The projected symmetric matrix with eigenvalues in the specified range.
    """
    # Step 1: Symmetrize the matrix
    sym_matrix = (matrix + matrix.T) / 2
    
    # Step 2: Perform eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(sym_matrix)
    
    # Step 3: Clip the eigenvalues to the specified range [eigen_min, eigen_max]
    clipped_eigenvalues = np.clip(eigenvalues, eigen_min, eigen_max)
    
    # Step 4: Reconstruct the matrix using the clipped eigenvalues
    projected_matrix = eigenvectors @ np.diag(clipped_eigenvalues) @ eigenvectors.T
    
    return projected_matrix

def project_to_simplex(y, a):
    u = np.flip(np.sort(y))
    k = 0
    while k <= len(y)-1:
        tau = (np.sum(u[:k+1]) - a)/(k+1) 
        if tau <= u[k]:
            k += 1
        else:
            break
    tau = (np.sum(u[:k]) - a)/(k) 
    out = y -tau
    out[out < 0] = 0 
    return out


In [127]:
#### Parameters
samples_per_cluster = [4, 3, 2]             # Number of training samples in each cluster
feature_dim = 5                             # Dimension of the features at each node
rho_min = 2                                 # Minimum eigenvalue of Ri
rho_max = 3                                 # Maximum eigenvalue of Ri
noise_level_Ri = 0.0                        # Scale of noise to adjust eigenvalues
include_rhos = True                         # Whether rho_min and rho_max are exact eigenvalues
num_clusters = len(samples_per_cluster)     # Number of clusters for clustering 
num_classes = num_clusters + 0              # Number of classes for classification
mu_0 = 1                                    # Regularization parameter
mu_1 = np.sum(samples_per_cluster)          # Regularization parameter
mu_2 = np.sum(samples_per_cluster)          # Regularization parameter

#### Create the graph and the data based on our model expressions

## Set up an adjacency matrix with num_clusters perfect clusters
A, A_bar, eigenvalues, E = create_adjacancy__matrix(samples_per_cluster)

#### write the code for other graphs 
#### ???????????????

## Generate symmetric matrices Ri
Ri_matrices = np.stack([create_Ri_symmetric_matrix_with_constrained_eigenvalues(feature_dim, rho_min, rho_max, noise_level=noise_level_Ri, include_extremes=include_rhos)
    for _ in range(num_clusters)], axis=0)

## Calculate Rv based on the provided expression
Rv_matrices = np.einsum("ij,jkn->ikn", ((E**2) @ np.diag(eigenvalues)), Ri_matrices)

## Generate feature vectors X based on the multivariate normal distribution
X = np.array([np.random.multivariate_normal(mean=np.zeros(feature_dim), cov=Rv, size=1).flatten() 
    for Rv in Rv_matrices])

## Generate the probability vector matrix b and generate matrix pi based on the provided expression
b = create_multiple_noisy_probability_vectors(num_clusters, num_classes, noise_level=0.0)
pi = ((E**2) @ np.diag(eigenvalues)) @ b

## Labels y based on the probability distributions in pi
y = np.array([np.random.choice(num_classes, p=pi_row) for pi_row in pi])

## Output for verification
# print("E Matrix:\n", E)
# print("\nPi Matrix:\n", pi)

In [143]:
#### The first step of COGENT: Finding the next atom, given the gradined of the function at (L_t, {{R_v}_v}_t, {{pi_kv}_kv}_t)
#### inputs: A_bar, {{R_v}_v}_t, {{pi_kv}_kv}_t, X, y, mu_1, mu_2, rho_min, rho_max, num_classes


#### computing the gradient: the dimesion of the first block does not match with other two blocks, so we only compute the blocks
## first block, gradient wrt L is always A_bar given as input
gradient_L = A_bar
## second block, gradinet wrt Rv is computed in the paper and can be computed as
Rv_matrices_inverse = np.array([np.linalg.inv(matrix) for matrix in Rv_matrices])
x_xtranspose = np.array([np.outer(xv, xv) for xv in X])
gradient_Rv = mu_1*np.array([np.matmul(np.matmul(Rv_matrices_inverse[i], x_xtranspose[i]), Rv_matrices_inverse[i])+np.eye(X.shape[1]) 
                                          for i in range(Rv_matrices_inverse.shape[0])])
# .reshape(Rv_matrices.shape[0]*Rv_matrices.shape[1],vRv_matrices.shape[2])
## third block, gradient wrt pi_k,v is computed in the paper and can be computed as
num_rows = y.size
num_cols = y.max() + 1
tmp = np.zeros((num_rows, num_cols), dtype=int)
tmp[np.arange(num_rows), y] = 1
gradient_pi = mu_2*tmp.flatten()

#### find the next atom by alternating minimization
## Initialization
iter_or_cycles_to_find_next_atom = 40
epsilon = np.random.randn(A_bar.shape[0])
R = create_Ri_symmetric_matrix_with_constrained_eigenvalues(Rv_matrices.shape[1], rho_min, rho_max, noise_level=0.0, include_extremes=True)
b = create_multiple_noisy_probability_vectors(1, num_classes, noise_level=1)[0]
## Alternating minimization
for it in range(iter_or_cycles_to_find_next_atom):
    ## fix R and b, update epsilon
    tmp1 = np.array([np.trace(matrix) for matrix in gradient_Rv])
    tmp2 = np.array([b[label] for label in y])
    tmp3 = gradient_L + np.diag(tmp1) + mu_2*np.diag(tmp2)
    eigvecs, _, _ = np.linalg.svd(tmp3)
    epsilon = eigvecs[:, 0]
    ## fix epsilon and b, update R 
    iner_iter_GA_R = 10
    iner_step_GA_R = 0.1
    for _ in range(iner_iter_GA_R):
        # grad_R = np.sum(Rv_matrices@np.diag(epsilon**2), axis=0)
        grad_R = np.sum(Rv_matrices * (epsilon**2)[:, np.newaxis, np.newaxis], axis=0)
        grad_step_R = R + iner_step_GA_R*grad_R 
        R = project_to_set_S(grad_step_R, rho_min, rho_max)
    ## fix epsilon and R, update b
    iner_iter_GA_b = 10
    iner_step_GA_b = 0.1
    for _ in range(iner_iter_GA_R):
        unique_labels = np.unique(y)
        grad_b_tmp = []
        for label in unique_labels:
            # Get indices where y equals the current label
            indices = np.where(np.array(y) == label)[0]
            # Extract the corresponding elements from z and compute the squared values
            grad_b_tmp.append(np.sum(epsilon[indices] ** 2))
        grad_b = np.array(grad_b_tmp)[0]
        grad_step_b = b + iner_step_GA_b*grad_b 
        b = project_to_simplex(grad_step_b ,1) 

    





In [144]:
b

array([0.71280317, 0.00922119, 0.27797564])

In [145]:
epsilon

array([ 0.05090646,  0.07189464,  0.31758923,  0.14612619, -0.92905538,
       -0.04269689, -0.06218374,  0.02680351,  0.0217195 ])

In [149]:
R

array([[ 3.00e+00, -4.24e-16,  1.12e-16,  3.97e-16, -3.56e-16],
       [-3.78e-16,  3.00e+00, -1.65e-16,  5.48e-16, -5.03e-16],
       [ 1.68e-16, -2.13e-16,  3.00e+00,  5.53e-16, -4.58e-16],
       [ 4.11e-16,  6.39e-16,  5.53e-16,  3.00e+00, -2.85e-16],
       [-3.93e-16, -4.70e-16, -4.09e-16, -2.93e-16,  3.00e+00]])

In [148]:
import numpy as np

# Example array with very small values
array = np.array([1.23456789e-10, 2.34567891e-10, 3.45678912e-10])

# Set print options to display more precision
np.set_printoptions(precision=2, suppress=False)

# Print the array to see values more clearly
print("Array with Increased Precision Display:")
print(array)


Array with Increased Precision Display:
[1.23e-10 2.35e-10 3.46e-10]


In [150]:
eiga,_ = np.linalg.eigh(R)
eiga

array([3., 3., 3., 3., 3.])